# Ponderada: Análise de Sensibilidade em Métricas de Interface Digital

Nesta atividade, você vai analisar quais variáveis de uma interface digital têm maior impacto sobre a taxa de conversão.

A entrega deve ser feita neste notebook, com código, tabelas, gráficos e respostas curtas.

## Contexto

Uma equipe de produto quer decidir qual métrica de interface deve receber prioridade no próximo ciclo de melhoria.

Os dados representam observações diárias de um aplicativo de compras.

A métrica alvo é a taxa de conversão.

As variáveis de entrada são taxa de abandono do carrinho, profundidade média de scroll e tempo até o primeiro clique em produto.

## Preparação

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.precision", 3)

## Dados

Execute a célula abaixo para criar a base da atividade.

In [ ]:
rng = np.random.default_rng(42)
n_dias = 180

taxa_abandono = rng.normal(48, 8, n_dias).clip(25, 75)
profundidade_scroll = rng.normal(62, 12, n_dias).clip(25, 95)
tempo_primeiro_clique = rng.normal(7, 2.2, n_dias).clip(2, 15)

ruido = rng.normal(0, 0.35, n_dias)
taxa_conversao = (
    7.5
    - 0.055 * taxa_abandono
    + 0.026 * profundidade_scroll
    - 0.085 * tempo_primeiro_clique
    + ruido
).clip(0.5, 9.0)

df = pd.DataFrame({
    "data": pd.date_range("2026-01-01", periods=n_dias, freq="D"),
    "taxa_abandono_carrinho_pct": taxa_abandono,
    "profundidade_scroll_pct": profundidade_scroll,
    "tempo_primeiro_clique_s": tempo_primeiro_clique,
    "taxa_conversao_pct": taxa_conversao,
})

df.head()

## Parte 1: Exploração

Crie ao menos um gráfico ou tabela para investigar a relação entre as variáveis de entrada e a taxa de conversão.

In [ ]:
# Use esta célula para criar sua análise exploratória.

colunas_numericas = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct",
    "tempo_primeiro_clique_s",
    "taxa_conversao_pct",
]

df[colunas_numericas].corr()

In [ ]:
# Preencha com uma variável de entrada para visualizar.
# Use exatamente um dos nomes que aparecem em features.

variavel_x = ""

if variavel_x not in features:
    raise ValueError("Preencha variavel_x com uma variável da lista features.")

fig = px.scatter(
    df,
    x=variavel_x,
    y="taxa_conversao_pct",
    title="Relação com a taxa de conversão",
)
fig.show()

Escreva quais duas variáveis você escolheu para a análise de sensibilidade e justifique com evidências da exploração.

**Resposta:**

## Parte 2: Modelo

Ajuste o modelo abaixo para estimar a taxa de conversão a partir das variáveis de entrada.

In [ ]:
features = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct",
    "tempo_primeiro_clique_s",
]
target = "taxa_conversao_pct"

X = df[features].to_numpy()
y = df[target].to_numpy()

X_design = np.column_stack([np.ones(len(X)), X])

coeficientes, *_ = np.linalg.lstsq(X_design, y, rcond=None)

pred = X_design @ coeficientes
erro = y - pred

mae = np.mean(np.abs(erro))
rmse = np.sqrt(np.mean(erro ** 2))

pd.DataFrame({
    "métrica": ["MAE", "RMSE"],
    "valor": [mae, rmse],
})

Interprete o erro do modelo em relação à taxa de conversão.

**Resposta:**

## Parte 3: Análise de Sensibilidade

Calcule a sensibilidade para duas variáveis de entrada usando uma variação de 10%.

Use a fórmula: sensibilidade igual à variação percentual da saída dividida pela variação percentual da entrada.

In [ ]:
def prever_linha(linha):
    entrada = np.array([1] + [linha[feature] for feature in features])
    return float(entrada @ coeficientes)


linha_base = df[features].mean().to_dict()
saida_base = prever_linha(linha_base)

linha_base, saida_base

In [ ]:
# Preencha com duas variáveis escolhidas na Parte 1.
# Use exatamente os nomes que aparecem em features.

variaveis_escolhidas = []

if len(variaveis_escolhidas) != 2:
    raise ValueError("Preencha variaveis_escolhidas com duas variáveis da lista features.")

variaveis_invalidas = [v for v in variaveis_escolhidas if v not in features]

if variaveis_invalidas:
    raise ValueError(f"Variáveis fora de features: {variaveis_invalidas}")

variacao_entrada = 0.10

resultados = []

for variavel in variaveis_escolhidas:
    linha_cenario = linha_base.copy()
    valor_original = linha_base[variavel]
    valor_alterado = valor_original * (1 + variacao_entrada)
    linha_cenario[variavel] = valor_alterado

    saida_nova = prever_linha(linha_cenario)
    variacao_saida = (saida_nova - saida_base) / saida_base
    indice_sensibilidade = variacao_saida / variacao_entrada

    resultados.append({
        "variável": variavel,
        "valor_original": valor_original,
        "valor_alterado": valor_alterado,
        "saída_original": saida_base,
        "saída_nova": saida_nova,
        "variação_saida_pct": variacao_saida * 100,
        "índice_sensibilidade": indice_sensibilidade,
    })

tabela_sensibilidade = pd.DataFrame(resultados)
tabela_sensibilidade

Compare os índices de sensibilidade e indique qual variável tem maior impacto sobre a taxa de conversão.

Mostre o raciocínio: cite os valores da tabela e explique o que eles significam para a decisão.

**Resposta:**

## Parte 4: Decisão

Recomende uma ação de produto ou interface com base na análise.

Sua recomendação deve citar os números da tabela de sensibilidade.

**Resposta:**

Aponte uma limitação, risco ou hipótese da sua análise.

**Resposta:**

## Ao Além dos Aléns

Faça uma simulação de Monte Carlo para estimar como a taxa de conversão pode variar sob incerteza nas variáveis de entrada.

In [ ]:
# Use esta célula para sua simulação.

n_simulacoes = 1000

amostras = pd.DataFrame({
    "taxa_abandono_carrinho_pct": rng.normal(
        linha_base["taxa_abandono_carrinho_pct"], 5, n_simulacoes
    ).clip(25, 75),
    "profundidade_scroll_pct": rng.normal(
        linha_base["profundidade_scroll_pct"], 8, n_simulacoes
    ).clip(25, 95),
    "tempo_primeiro_clique_s": rng.normal(
        linha_base["tempo_primeiro_clique_s"], 1.5, n_simulacoes
    ).clip(2, 15),
})

amostras_design = np.column_stack([
    np.ones(len(amostras)),
    amostras[features].to_numpy(),
])
previsoes = amostras_design @ coeficientes

pd.Series(previsoes).describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])

In [ ]:
fig = px.histogram(
    pd.DataFrame({"taxa_conversao_pct_prevista": previsoes}),
    x="taxa_conversao_pct_prevista",
    nbins=30,
    title="Distribuição simulada da taxa de conversão",
)
fig.show()

Interprete o que a distribuição simulada indica sobre o risco da sua recomendação.

**Resposta:**

## Uso de IA

O uso de IA é permitido para apoio técnico, revisão de texto e estudo dos conceitos.

As escolhas de variáveis, os cálculos, a comparação dos índices e a recomendação devem refletir sua análise dos resultados deste notebook.

Você deve ser capaz de explicar qualquer resposta entregue.

Respostas sem relação com os números gerados, com indícios de cópia ou que não possam ser justificadas poderão ser tratadas como fora da proposta.

## Instruções de entrega

A entrega deverá ser feita no GitHub ou no próprio Google Colab.

Links sem permissão de acesso terão um desconto de 20% na nota.